In [ ]:
# =============================================================================
# HTR-10 Pebble Bed Reactor — OpenMC Model
# =============================================================================

import warnings
import openmc
from openmc import IDWarning
import numpy as np
import os
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=IDWarning)

os.chdir('/home/eric/Documents/openMC_models/PBR')
print(f"Working directory: {os.getcwd()}")
openmc.config['cross_sections'] = '/home/eric/openmc/nuclear_data/endf-b8.0-hdf5/endfb-viii.0-hdf5/cross_sections.xml'
os.environ['OPENMC_CROSS_SECTIONS'] = str(openmc.config['cross_sections'])



In [ ]:

# 'dev'        fast, rough k-eff, debugging
# 'portfolio'  hours, good statistics
MODE = 'portfolio'

CONFIG = {
    #                                                                        rod_insertion: 0=withdrawn, 1=fully inserted
    'dev':       dict(n_triso=500,   n_pebbles=3000,  n_particles=1000,  n_batches=100, n_inactive=30,  rod_insertion=0.7),
    'portfolio': dict(n_triso=8335,  n_pebbles=27000, n_particles=10000, n_batches=200, n_inactive=50,  rod_insertion=0.85),
}[MODE]

print(f"Mode: {MODE}")
print(f"  TRISO per pebble : {CONFIG['n_triso']}")
print(f"  Pebbles in core  : {CONFIG['n_pebbles']}")
print(f"  Particles/batch  : {CONFIG['n_particles']}")
print(f"  Batches (total)  : {CONFIG['n_batches']}  ({CONFIG['n_inactive']} inactive)")
print(f"  Rod insertion    : {CONFIG['rod_insertion']*100:.0f}% of active height")


In [ ]:

# =============================================================================
# Materials
# =============================================================================
#
# All densities and compositions from:
#   IAEA-TECDOC-1382, HTR-10 specification (2003)
#   Wu, Nuclear Engineering and Design 218 (2002) 25-32
# =============================================================================

# UO2 fuel kernel — 17 w/o enriched, 10.4 g/cm3
# Number densities (atoms/b-cm) computed from enrichment + density
uo2 = openmc.Material(name='UO2 Kernel')
uo2.add_nuclide('U235',  3.8228e-3)
uo2.add_nuclide('U238',  1.8612e-2)
uo2.add_nuclide('O16',   4.4870e-2)
uo2.set_density('atom/b-cm', 6.7100e-2)

# Buffer PyC — porous, low density, first coating layer
buffer_pyc = openmc.Material(name='Buffer PyC')
buffer_pyc.add_element('C', 1.0)
buffer_pyc.set_density('g/cm3', 1.05)

# Inner PyC
ipy_c = openmc.Material(name='Inner PyC')
ipy_c.add_element('C', 1.0)
ipy_c.set_density('g/cm3', 1.90)

# SiC pressure vessel layer
sic = openmc.Material(name='SiC')
sic.add_element('Si', 1.0, 'ao')
sic.add_element('C',  1.0, 'ao')
sic.set_density('g/cm3', 3.18)

# Outer PyC
opy_c = openmc.Material(name='Outer PyC')
opy_c.add_element('C', 1.0)
opy_c.set_density('g/cm3', 1.90)

# Graphite matrix surrounding TRISO particles inside the fuel zone
matrix_graphite = openmc.Material(name='Matrix Graphite')
matrix_graphite.add_element('C', 1.0)
matrix_graphite.set_density('g/cm3', 1.73)

# Pebble outer shell (pure graphite annulus, 0.5 cm thick)
shell_graphite = openmc.Material(name='Shell Graphite')
shell_graphite.add_element('C', 1.0)
shell_graphite.set_density('g/cm3', 1.73)

# Helium coolant — 7 MPa, ~250 C inlet conditions
helium = openmc.Material(name='Helium Coolant')
helium.add_element('He', 1.0)
helium.set_density('g/cm3', 0.000164)

# Graphite reflector (side + top + bottom blocks)
# Slightly higher density than matrix/shell — reflector-grade graphite
reflector = openmc.Material(name='Reflector Graphite')
reflector.add_element('C', 1.0)
reflector.set_density('g/cm3', 1.76)

# Operating temperatures — full-power hot conditions (IAEA-TECDOC-1382)
# Fuel/coatings run hottest; reflector and coolant run cooler.
uo2.temperature             = 1000  # K — drives Doppler feedback
buffer_pyc.temperature      =  900
ipy_c.temperature           =  900
sic.temperature             =  900
opy_c.temperature           =  900
matrix_graphite.temperature =  900
shell_graphite.temperature  =  850
helium.temperature          =  750  # K — avg of 523 K inlet / 973 K outlet
reflector.temperature       =  600  # K — reflector runs significantly cooler than core

materials = openmc.Materials([
    uo2, buffer_pyc, ipy_c, sic, opy_c,
    matrix_graphite, shell_graphite, helium, reflector
])
materials.export_to_xml()
print("materials.xml written")


In [ ]:

# =============================================================================
# Pebble Universe
# =============================================================================
#
# Uses a homogenized UO2+graphite fuel zone (volume weighted mix).
# Pebble: r=3.0 cm total, fuel zone r=2.5 cm, graphite shell 0.5 cm
# =============================================================================

r_opy_c     = 0.0455   # TRISO outer radius (for volume fraction calc)
r_pebble    = 3.0
r_fuel_zone = 2.5

# Volume weighted mix using TRISO count 8335
# triso_vf: fraction of fuel zone occupied by full TRISO particles
# uo2_vf:   fraction occupied by UO2 kernel only — kernel is r=0.025, TRISO outer r=0.0455
triso_vf = 8335 * (r_opy_c    / r_fuel_zone)**3
uo2_vf   = 8335 * (0.0250     / r_fuel_zone)**3
fuel_mix = openmc.Material(name='Homogenized Fuel')
c12_ad = matrix_graphite.get_nuclide_atom_densities()['C12']
fuel_mix.add_nuclide('U235', 3.8228e-3 * uo2_vf)
fuel_mix.add_nuclide('U238', 1.8612e-2 * uo2_vf)
fuel_mix.add_nuclide('O16',  4.4870e-2 * uo2_vf)
fuel_mix.add_nuclide('C12',  c12_ad * (1 - triso_vf))
fuel_mix.set_density('atom/b-cm',
    (3.8228e-3 + 1.8612e-2 + 4.4870e-2) * uo2_vf + c12_ad * (1 - triso_vf))
fuel_mix.temperature = 900  # K — homogenized fuel zone average
materials.append(fuel_mix)
materials.export_to_xml()

s_fuel_zone = openmc.Sphere(r=r_fuel_zone)
s_pebble    = openmc.Sphere(r=r_pebble)
pebble_univ = openmc.Universe(name='Fuel Pebble', cells=[
    openmc.Cell(fill=fuel_mix,       region=-s_fuel_zone,             name='fuel zone'),
    openmc.Cell(fill=shell_graphite, region=+s_fuel_zone & -s_pebble, name='shell'),
])
print('Pebble universe built (homogenized)')

r_core     =  90.0
h_half     =  98.5
r_ref_out  = 170.0
z_ref_top  = +198.5
z_ref_bot  = -198.5

core_cyl  = openmc.ZCylinder(r=r_core)
core_top  = openmc.ZPlane(z0=+h_half)
core_bot  = openmc.ZPlane(z0=-h_half)
ref_cyl   = openmc.ZCylinder(r=r_ref_out,  boundary_type='vacuum')
ref_top   = openmc.ZPlane(z0=z_ref_top,    boundary_type='vacuum')
ref_bot   = openmc.ZPlane(z0=z_ref_bot,    boundary_type='vacuum')

pebble_cache = f'pebble_centers_{CONFIG["n_pebbles"]}.npy'
if os.path.exists(pebble_cache):
    pebble_centers = np.load(pebble_cache)
    print(f"  Loaded {len(pebble_centers)} pebble centers from cache")
else:
    print(f"Packing {CONFIG['n_pebbles']} pebbles...")
    pf_pebble = CONFIG['n_pebbles'] * (4/3 * np.pi * r_pebble**3) / (np.pi * r_core**2 * 2 * h_half)
    pebble_centers = openmc.model.pack_spheres(
        radius=r_pebble,
        region=-core_cyl & +core_bot & -core_top,
        pf=min(pf_pebble, 0.60),
        num_spheres=CONFIG['n_pebbles']
    )
    np.save(pebble_cache, pebble_centers)
    print(f"  Packed and cached {len(pebble_centers)} pebble centers")

# Build TRISO objects for each pebble
pebble_trisos = [
    openmc.model.TRISO(outer_radius=r_pebble, fill=pebble_univ, center=c)
    for c in pebble_centers
]

# create_triso_lattice sorts pebbles into a background lattice for faster transport
llc = np.array([-r_core, -r_core, -h_half])
urc = np.array([ r_core,  r_core,  h_half])

core_lat_univ = openmc.model.create_triso_lattice(
    pebble_trisos,
    llc,
    urc,
    [10, 10, 20],
    helium
)

core_cell = openmc.Cell(fill=core_lat_univ, region=-core_cyl & +core_bot & -core_top, name='core')

# =============================================================================
# Control rods — 10 B4C absorber rods in side reflector
# HTR-10 spec: rods at r=105 cm, symmetric, r_rod=6 cm
# rod_insertion: fraction of active core height (2*h_half) inserted from top
# =============================================================================
b4c = openmc.Material(name='B4C Absorber')
b4c.add_element('B', 4.0, 'ao')
b4c.add_element('C', 1.0, 'ao')
b4c.set_density('g/cm3', 2.52)
b4c.temperature = 600  # K — rods sit in reflector, runs cooler than core
materials.append(b4c)
materials.export_to_xml()

n_rods      = 10
r_rod_ctr   = 105.0   # radial position of rod centerline in reflector (cm)
r_rod       = 6.0     # rod channel radius (cm)
insertion   = CONFIG['rod_insertion']
z_rod_tip   = h_half - insertion * 2 * h_half   # z of bottom of inserted absorber
rod_top     = core_top                           # rods flush with active core top
rod_tip     = openmc.ZPlane(z0=z_rod_tip)        # bottom of B4C column

rod_cells = []
angles = np.linspace(0, 2*np.pi, n_rods, endpoint=False)
for i, theta in enumerate(angles):
    cx = r_rod_ctr * np.cos(theta)
    cy = r_rod_ctr * np.sin(theta)
    rod_cyl = openmc.ZCylinder(x0=cx, y0=cy, r=r_rod)
    # Absorber column: from rod tip up to core top
    absorber = openmc.Cell(
        fill=b4c, region=-rod_cyl & +rod_tip & -rod_top,
        name=f'rod_{i}_absorber')
    # Gas plenum below tip (helium — rod not present here)
    plenum = openmc.Cell(
        fill=helium, region=-rod_cyl & +ref_bot & -rod_tip,
        name=f'rod_{i}_plenum')
    # Gas space above core top (withdrawn portion above active zone)
    above = openmc.Cell(
        fill=helium, region=-rod_cyl & +rod_top & -ref_top,
        name=f'rod_{i}_above')
    rod_cells.extend([absorber, plenum, above])

# Side reflector: graphite minus all rod channels
rod_cyls = [openmc.ZCylinder(x0=r_rod_ctr*np.cos(t), y0=r_rod_ctr*np.sin(t), r=r_rod)
            for t in angles]
# 'outside all rod cylinders' = intersection of all complements
from functools import reduce
no_rods = reduce(lambda a, b: a & b, [+c for c in rod_cyls])
side_ref = openmc.Cell(fill=reflector,
    region=+core_cyl & -ref_cyl & +ref_bot & -ref_top & no_rods,
    name='side reflector')
top_ref  = openmc.Cell(fill=reflector, region=-ref_cyl & +core_top & -ref_top,  name='top reflector')
bot_ref  = openmc.Cell(fill=reflector, region=-ref_cyl & +ref_bot  & -core_bot, name='bottom reflector')

root_univ = openmc.Universe(cells=[core_cell, side_ref, top_ref, bot_ref] + rod_cells)
geometry  = openmc.Geometry(root_univ)
geometry.export_to_xml()
print(f"geometry.xml written  (rods {insertion*100:.0f}% inserted, tip at z={z_rod_tip:.1f} cm)")


In [ ]:

# =============================================================================
# Settings
# =============================================================================

settings = openmc.Settings()
settings.batches   = CONFIG['n_batches']
settings.inactive  = CONFIG['n_inactive']
settings.particles = CONFIG['n_particles']
settings.run_mode  = 'eigenvalue'

# Cross-section temperature interpolation — required when material.temperature not 293 K.
# interpolation linearly interpolates between the two nearest tabulated temps
# in the HDF5 library (250 K, 600 K, 900 K, 1200 K, 2500 K for ENDF/B-VIII.0).
settings.temperature = {'method': 'interpolation', 'range': [250, 2500]}

# scatter points randomly WITHIN each pebble's fuel zone radius
# to maximize chance of hitting a TRISO kernel

rng = np.random.default_rng(13)

# Source at pebble centers: neutrons thermalize in graphite then cause fission.

n_src = min(1000, len(pebble_centers))
chosen = pebble_centers[rng.choice(len(pebble_centers), size=n_src, replace=False)]
settings.source = [
    openmc.IndependentSource(space=openmc.stats.Point((float(c[0]), float(c[1]), float(c[2]))))
    for c in chosen
]
print(f"Source: {len(settings.source)} points at pebble centers")

settings.export_to_xml()
print("settings.xml written")



# =============================================================================
# Tallies
# =============================================================================

tallies = openmc.Tallies()

# Axial mesh — 20 slices over active core height
axial_mesh           = openmc.RegularMesh(name='axial')
axial_mesh.dimension = [1, 1, 20]
axial_mesh.lower_left  = [-r_core, -r_core, -h_half]
axial_mesh.upper_right = [ r_core,  r_core,  h_half]

axial_tally = openmc.Tally(name='axial_flux')
axial_tally.filters = [openmc.MeshFilter(axial_mesh)]
axial_tally.scores  = ['flux', 'fission']
tallies.append(axial_tally)

# Cylindrical radial mesh — 15 radial bins
radial_mesh = openmc.CylindricalMesh(
    r_grid   = np.linspace(0, r_core, 16),
    z_grid   = [-h_half, h_half],
    phi_grid = [0, 2 * np.pi],
    name     = 'radial'
)

radial_tally = openmc.Tally(name='radial_flux')
radial_tally.filters = [openmc.MeshFilter(radial_mesh)]
radial_tally.scores  = ['flux', 'fission']
tallies.append(radial_tally)

# Broad energy spectrum (6 groups, thermal → fast)
energy_tally = openmc.Tally(name='energy_spectrum')
energy_tally.filters = [openmc.EnergyFilter([0.0, 0.625, 5.53e3, 1.0e5, 1.0e6, 8.0e6, 2.0e7])]
energy_tally.scores  = ['flux']
tallies.append(energy_tally)

# 2D r-z flux map — cylindrical mesh over active core + reflector
# 25 radial bins (0→170 cm covers core + side reflector) × 40 axial bins
# Two energy groups: thermal (E < 0.625 eV) and fast (E > 0.625 eV)
rz_mesh = openmc.CylindricalMesh(
    r_grid   = np.linspace(0, r_ref_out, 26),   # 25 bins, core+reflector radially
    z_grid   = np.linspace(-h_half, h_half, 41), # 40 bins over active height
    phi_grid = [0, 2 * np.pi],
    name     = 'rz_map'
)
rz_tally = openmc.Tally(name='rz_flux_map')
rz_tally.filters = [
    openmc.MeshFilter(rz_mesh),
    openmc.EnergyFilter([0.0, 0.625, 2.0e7]),   # group 0=thermal, group 1=fast
]
rz_tally.scores = ['flux']
tallies.append(rz_tally)

# Shannon entropy mesh
entropy_mesh = openmc.RegularMesh()
entropy_mesh.lower_left  = [-r_core, -r_core, -h_half]
entropy_mesh.upper_right = [ r_core,  r_core,  h_half]
entropy_mesh.dimension   = [10, 10, 20]
settings.entropy_mesh = entropy_mesh

settings.export_to_xml()
tallies.export_to_xml()
print("settings.xml written")
print("tallies.xml written")


In [ ]:
geom = openmc.Geometry.from_xml()
mat_by_name = {m.name: m for m in geom.get_all_materials().values()}

fuel_name = 'Homogenized Fuel' if 'Homogenized Fuel' in mat_by_name else 'UO2 Kernel'
colors = {
    mat_by_name[fuel_name]:            (166,  25,  25),
    mat_by_name['Shell Graphite']:     (140, 140, 140),
    mat_by_name['Helium Coolant']:     (135, 206, 250),
    mat_by_name['Reflector Graphite']: (217, 217, 190),
}

plot_kw = dict(color_by='material', colors=colors)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

geom.plot(origin=(0, 0, 0), width=(360, 420), pixels=(540, 630),
          basis='xz', axes=axes[0], **plot_kw)
axes[0].set_title('Full reactor — XZ')
axes[0].set_xlabel('x (cm)')
axes[0].set_ylabel('z (cm)')

geom.plot(origin=(0, 0, 0), width=(195, 210), pixels=(400, 430),
          basis='xz', axes=axes[1], **plot_kw)
axes[1].set_title('Active core — XZ')
axes[1].set_xlabel('x (cm)')

geom.plot(origin=(0, 0, 0), width=(230, 230), pixels=(400, 400),
          basis='xy', axes=axes[2], **plot_kw)
axes[2].set_title('Active core — XY (z=0)')
axes[2].set_xlabel('x (cm)')
axes[2].set_ylabel('y (cm)')

fig.suptitle('HTR-10 PBR Geometry', fontsize=13)
fig.tight_layout()
fig.savefig('geometry.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:

openmc.run()

In [ ]:
# =============================================================================
# 2D r-z Flux Map
# =============================================================================
import matplotlib.lines as mlines

SP_FILE = f'statepoint.{CONFIG["n_batches"]}.h5'

with openmc.StatePoint(SP_FILE) as sp:
    t = sp.get_tally(name='rz_flux_map')

    mesh_filter   = t.find_filter(openmc.MeshFilter)
    energy_filter = t.find_filter(openmc.EnergyFilter)
    mesh = mesh_filter.mesh

    n_r = len(mesh.r_grid) - 1
    n_z = len(mesh.z_grid) - 1
    n_e = len(energy_filter.bins)

    flux_raw = t.get_values(scores=['flux']).squeeze()
    flux = flux_raw.reshape(n_z, n_r, n_e)

    thermal = flux[:, :, 0]
    fast    = flux[:, :, 1]

r_edges = mesh.r_grid
z_edges = mesh.z_grid

thermal_norm = thermal / thermal.max()
fast_norm    = fast    / fast.max()
total = thermal + fast
with np.errstate(invalid='ignore', divide='ignore'):
    th_frac = np.where(total > 0, thermal / total, np.nan)

z_tip  = 98.5 - CONFIG['rod_insertion'] * 2 * 98.5
r_core = 90.0

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

def add_annotations(ax):
    ax.axvline(r_core, color='black', lw=1.5, ls='-')
    ax.axhline(z_tip,  color='tab:red', lw=1.5, ls='--')

line_core = mlines.Line2D([], [], color='black',   lw=1.5, ls='-',  label='Core edge  r = 90 cm')
line_rod  = mlines.Line2D([], [], color='tab:red', lw=1.5, ls='--', label=f'Rod tip  z = {z_tip:.0f} cm')
legend_kw = dict(handles=[line_core, line_rod], fontsize=9, loc='lower right')

# --- Panel 1: Thermal flux ---
ax = axes[0]
im = ax.pcolormesh(r_edges, z_edges, thermal_norm, cmap='inferno', vmin=0, vmax=1)
fig.colorbar(im, ax=ax, label='Normalized flux')
add_annotations(ax)
ax.set_xlabel('Radius r (cm)')
ax.set_ylabel('Axial position z (cm)')
ax.set_title('Thermal flux  (E < 0.625 eV)')
ax.legend(**legend_kw)

# --- Panel 2: Fast flux ---
ax = axes[1]
im = ax.pcolormesh(r_edges, z_edges, fast_norm, cmap='viridis', vmin=0, vmax=1)
fig.colorbar(im, ax=ax, label='Normalized flux')
add_annotations(ax)
ax.set_xlabel('Radius r (cm)')
ax.set_title('Fast flux  (E > 0.625 eV)')

# --- Panel 3: Thermal fraction ---
ax = axes[2]
im = ax.pcolormesh(r_edges, z_edges, th_frac, cmap='RdYlBu', vmin=0, vmax=1)
fig.colorbar(im, ax=ax, label='Thermal fraction')
add_annotations(ax)
ax.set_xlabel('Radius r (cm)')
ax.set_title('Thermal fraction  φ_th / (φ_th + φ_fast)')

fig.suptitle(
    f'HTR-10 r-z Flux Map',
    fontsize=13
)
fig.tight_layout()
fig.savefig('flux_map_rz.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved flux_map_rz.png")


In [ ]:
# =============================================================================
# Source Convergence
# =============================================================================

SP_FILE = f'statepoint.{CONFIG["n_batches"]}.h5'

with openmc.StatePoint(SP_FILE) as sp:
    k_gen      = np.array(sp.k_generation)
    n_inactive = sp.n_inactive
    n_batches  = sp.n_batches

batches = np.arange(1, n_batches + 1)

# Running mean and std of k over active batches only
active_k = k_gen[n_inactive:]
active_b = batches[n_inactive:]
running_mean = np.cumsum(active_k) / np.arange(1, len(active_k) + 1)
running_std  = np.array([active_k[:i].std() for i in range(1, len(active_k) + 1)])

fig, ax2 = plt.subplots(figsize=(11, 7))
fig.suptitle('HTR-10 Source Convergence', fontsize=13)

# --- Bottom: k per generation + running mean ± 1sigma ---
ax2.plot(batches[:n_inactive], k_gen[:n_inactive], color='tab:gray', lw=0.6, alpha=0.6, label='k/gen (inactive)')
ax2.plot(batches[n_inactive:], k_gen[n_inactive:], color='tab:blue', lw=0.6, alpha=0.6, label='k/gen (active)')
ax2.plot(active_b, running_mean, color='tab:red', lw=1.8, label='Running mean (active)')
ax2.fill_between(active_b,
                 running_mean - running_std,
                 running_mean + running_std,
                 color='tab:red', alpha=0.15, label='±1σ')
ax2.axvline(n_inactive + 0.5, color='black', lw=1.0, ls='--')
ax2.axhline(running_mean[-1], color='tab:red', lw=0.8, ls=':', alpha=0.6)
ax2.set_ylabel('k-effective')
ax2.set_xlabel('Batch')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# Annotate final k ± std on the plot
ax2.annotate(
    f'  k = {running_mean[-1]:.5f} ± {running_std[-1]:.5f}',
    xy=(active_b[-1], running_mean[-1]),
    xytext=(n_inactive + 5, running_mean[-1]),
    fontsize=9, color='tab:red',
    va='center'
)

fig.tight_layout()
fig.savefig('convergence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# Doppler Coefficient Study
# Restarts from the converged statepoint.200.h5 so each
# temperature point needs only a small inactive buffer instead of ~50 batches.
# Only fuel zone temperature varies — moderator/reflector held fixed to isolate
# the Doppler effect from the moderator temperature coefficient.
# =============================================================================

from scipy.stats import linregress

STATEPOINT  = 'statepoint.200.h5'   # converged source from the portfolio run
FUEL_TEMPS  = [600, 750, 900, 1050, 1200]  # K
N_ACTIVE    = 50
N_INACTIVE  = 5    # small buffer — source shape shifts slightly with temperature
N_PARTICLES = 10000

keff_vals  = []
keff_sigma = []

for T_fuel in FUEL_TEMPS:
    print(f"\n--- T_fuel = {T_fuel} K ---", flush=True)

    # Only update fuel zone temperatures. reflector/helium/B4C stay nominal.
    # the active fuel material is fuel_mix
    fuel_mix.temperature        = T_fuel
    uo2.temperature             = T_fuel
    buffer_pyc.temperature      = T_fuel
    ipy_c.temperature           = T_fuel
    sic.temperature             = T_fuel
    opy_c.temperature           = T_fuel
    matrix_graphite.temperature = T_fuel
    shell_graphite.temperature  = T_fuel
    materials.export_to_xml()

    cfg = openmc.Settings()
    cfg.batches     = N_INACTIVE + N_ACTIVE
    cfg.inactive    = N_INACTIVE
    cfg.particles   = N_PARTICLES
    cfg.run_mode    = 'eigenvalue'
    cfg.temperature = {'method': 'interpolation', 'range': [250, 2500]}
    cfg.entropy_mesh = entropy_mesh
    # Pre-converged fission source eliminates the long inactive calcs
    cfg.source = openmc.FileSource(STATEPOINT)
    cfg.export_to_xml()

    openmc.run(output=False)

    sp_file = f'statepoint.{N_INACTIVE + N_ACTIVE}.h5'
    with openmc.StatePoint(sp_file) as sp:
        k   = float(sp.keff.nominal_value)
        sig = float(sp.keff.std_dev)
    keff_vals.append(k)
    keff_sigma.append(sig)
    print(f"  k-eff = {k:.5f} ± {sig:.5f}")

# Restore nominal material temperatures and settings for subsequent cells
fuel_mix.temperature        = 900
uo2.temperature             = 1000
buffer_pyc.temperature      = 900
ipy_c.temperature           = 900
sic.temperature             = 900
opy_c.temperature           = 900
matrix_graphite.temperature = 900
shell_graphite.temperature  = 850
materials.export_to_xml()
settings.export_to_xml()
print("\nNominal temperatures set")

# =============================================================================
# Doppler coefficient: linear fit of k vs T_fuel -> alpha_D in pcm/K
# alpha_D = (dk/dT) / k_ref x 1e5
# HTR-10 reference: −3 to −5 pcm/K (IAEA-TECDOC-1382)
# =============================================================================

temps  = np.array(FUEL_TEMPS,  dtype=float)
keffs  = np.array(keff_vals,   dtype=float)
sigmas = np.array(keff_sigma,  dtype=float)

slope, intercept, r_val, _, _ = linregress(temps, keffs)
k_ref   = keffs[FUEL_TEMPS.index(900)]
alpha_D = slope / k_ref * 1e5   # pcm/K

print(f"\nDoppler coefficient  alpha_D = {alpha_D:.2f} pcm/K")
print(f"Linear fit R²            = {r_val**2:.4f}")
print(f"HTR-10 reference range   : −3 to −5 pcm/K  (IAEA-TECDOC-1382)")


In [ ]:
temps  = np.array(FUEL_TEMPS,  dtype=float)
keffs  = np.array(keff_vals,   dtype=float)
sigmas = np.array(keff_sigma,  dtype=float)

slope, intercept, r_val, _, _ = linregress(temps, keffs)
k_ref   = keffs[FUEL_TEMPS.index(900)]
alpha_D = slope / k_ref * 1e5

t_fit = np.linspace(temps[0] - 50, temps[-1] + 50, 300)
k_fit = slope * t_fit + intercept

rho     = (keffs - 1.0) / keffs * 1e5
drho    = 2.0 * sigmas / keffs**2 * 1e5
rho_fit = (k_fit  - 1.0) / k_fit  * 1e5

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('HTR-10 Doppler Reactivity Coefficient', fontsize=13)

ax1.plot(t_fit, k_fit, '--', color='tab:gray', lw=1.2, label='Linear fit')
ax1.errorbar(temps, keffs, yerr=2*sigmas, fmt='o', color='tab:red',
             capsize=4, markersize=6, label='OpenMC (2σ)')
ax1.axhline(1.0, color='black', lw=0.8, ls=':')
ax1.set_xlabel('Fuel Temperature (K)')
ax1.set_ylabel('k-effective')
ax1.set_title('k-eff vs Fuel Temperature')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.plot(t_fit, rho_fit, '--', color='tab:gray', lw=1.2)
ax2.errorbar(temps, rho, yerr=drho, fmt='o', color='tab:red',
             capsize=4, markersize=6)
ax2.axhline(0.0, color='black', lw=0.8, ls=':')
ax2.set_xlabel('Fuel Temperature (K)')
ax2.set_ylabel('Reactivity ρ (pcm)')
ax2.set_title(f'Doppler Coefficient  α_D = {alpha_D:.2f} pcm/K')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig('doppler_coefficient.png', dpi=150, bbox_inches='tight')
plt.show()
